# Surreal ORM Lite v0.14.3 — correctness fixes (issue #156)

> A correctness release. Eight findings from
> [issue #156](https://github.com/EulogySnowfall/SurrealDB-ORM-lite/issues/156) — seven reported,
> one found while fixing another — each reproduced against a live SurrealDB **3.2.4 and 2.6.5**
> before the fix. **Everything below behaves identically on both DB lines**, so there is no
> capability probe in this notebook.

The one new piece of API is `Var(...)`, plus the `"$$literal"` escape. The rest is behaviour
that was wrong and is now right; each section shows what the old code did, in the prose, and
what runs today, in the output.


## 1. Connect


In [1]:
import os

from surreal_orm_lite import SurrealDBConnectionManager

HOST = os.environ.get("SURREALDB_HOST", "localhost")
PORT = os.environ.get("SURREALDB_PORT", "8000")

SurrealDBConnectionManager.set_connection(
    url=f"ws://{HOST}:{PORT}/rpc",
    user="root",
    password="root",
    namespace="examples",
    database="v0143",
)
print("Connection configured:", SurrealDBConnectionManager.is_connection_set())


Connection configured: True


## 2. The models

`Account` is an ordinary model. `Flexible` opts into undeclared fields with
`extra="allow"` — section 6 shows why that matters.


In [2]:
from pydantic import ConfigDict, Field
from surrealdb import RecordID

from surreal_orm_lite import BaseSurrealModel


class Account(BaseSurrealModel):
    id: str | RecordID | None = None
    name: str = Field(...)
    plan: str = "free"
    seats: int = 1


class Flexible(BaseSurrealModel):
    model_config = ConfigDict(extra="allow")

    id: str | RecordID | None = None
    name: str = Field(...)


client = await SurrealDBConnectionManager.get_client()
for table in ("Account", "Flexible", "Membership"):
    try:
        await client.query(f"REMOVE TABLE {table};")
    except Exception:
        pass

for account in (
    Account(id="ada", name="Ada", plan="pro", seats=3),
    Account(id="grace", name="Grace", seats=2),
    Account(id="linus", name="Linus", seats=1),
):
    await account.save()

print("seeded:", len(await Account.objects().exec()), "accounts")


seeded: 3 accounts


## 3. `first()` leaves the queryset alone

It used to assign `LIMIT 1` to the queryset and never take it back, so the *next* use of the
same queryset quietly returned at most one row.


In [3]:
qs = Account.objects().filter(seats__gte=1)

print("first():", (await qs.first()).name)
print("then exec():", [a.name for a in await qs.exec()])
print("limit left behind:", qs._limit)


first(): Ada
then exec(): ['Ada', 'Grace', 'Linus']
limit left behind: None


## 4. Filter values that start with `$`

A string beginning with `$` is read as a **query variable reference**, which makes a literal
like `"$admin"` unmatchable and turns user input into an accidental reference. Say it
explicitly with `Var(...)`, and escape a real dollar sign by doubling it.


In [4]:
from surreal_orm_lite import Var

await Account(id="dollar", name="$admin", seats=1).save()

# "$$admin" → the literal string "$admin"
literal = await Account.objects().filter(name="$$admin").exec()
print("literal match:", [a.name for a in literal])

# Var("…") → an explicit reference to a bound query variable
qs = Account.objects().filter(seats__gte=Var("min_seats"))
query, _ = qs._compile_query()
print("compiled  :", query.strip())


literal match: ['$admin']
compiled  : SELECT * FROM Account WHERE seats >= $min_seats;


The bare `"$x"` form still works — it is what the ORM has always done — but it now warns,
so the default can be flipped in a later minor without breaking anyone quietly.


In [5]:
import warnings

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    Account.objects().filter(name="$admin")._compile_query()

print(caught[0].category.__name__)
print(str(caught[0].message)[:120], "…")


DeprecationWarning
Passing '$admin' as a filter value is interpreted as a reference to the query variable $admin and is deprecated; use Var …


## 5. Your raw SurrealQL is sent verbatim

`raw_query()` used to rewrite quoted `'$word'` occurrences *inside your own string literals*,
so a `WHERE name = '$admin'` searched for an unbound variable and found nothing.


In [6]:
rows = await Account.raw_query("SELECT * FROM Account WHERE name = '$admin';")
print("raw_query found:", [r.name for r in rows])


raw_query found: ['$admin']


A multi-statement query now warns, because the SDK returns only the first statement's rows:


In [7]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    await Account.raw_query("SELECT name FROM Account; SELECT seats FROM Account;")

print(str(caught[0].message))


The query carries 2 statements but only the first one's results are returned by the SurrealDB SDK; run one statement per call, or use transaction() if they must be atomic.


## 6. `update_or_create()` refuses keys the model does not declare

They used to be dropped by Pydantic when creating, and written straight into the row when
updating — so the same call produced two different stored schemas depending on whether the
record already existed.


In [8]:
from surreal_orm_lite.exceptions import SurrealDbError

try:
    await Account.objects().update_or_create(defaults={"setas": 9}, name="Ada")
except SurrealDbError as error:
    print("refused:", error)

account, created = await Account.objects().update_or_create(defaults={"seats": 9}, name="Ada")
print("declared key:", account.name, account.seats, "| created:", created)


refused: Account does not declare setas; remove the key, add the field to the model, or set model_config = ConfigDict(extra="allow") to store undeclared fields.
declared key: Ada 9 | created: False


A model that opts into extra fields keeps them, on both paths:


In [9]:
flexible, created = await Flexible.objects().update_or_create(defaults={"nickname": "gigi"}, name="Grace")
print("extra kept:", flexible.name, getattr(flexible, "nickname", None), "| created:", created)


extra kept: Grace gigi | created: True


## 7. `upsert()` reports `created` truthfully

`upsert()` is a full REPLACE (unchanged), but the `post_save` signal used to claim a creation
every single time. The flag now comes from the statement's `$before`, at no extra round-trip.


In [10]:
from surreal_orm_lite import post_save

seen = []


@post_save.connect(Account)
async def _record(sender, instance, created, **kwargs):
    seen.append((instance.name, created))


await Account(id="tim", name="Tim", seats=1).upsert()  # new record
await Account(id="tim", name="Tim II", seats=4).upsert()  # replaces it

print("post_save saw:", seen)


post_save saw: [('Tim', True), ('Tim II', False)]


## 8. Relations: one round-trip, and numeric-looking ids work

`get_related(model_class=…)` projects the records inside the traversal instead of fetching ids
and selecting them afterwards. And an id like `"1"` is stored as the *string* record id, so it
is now backtick-quoted — unquoted, `Account:1` is the *integer* id, a record the ORM never
wrote, and every relation on it silently traversed nothing.


In [11]:
numeric = Account(id="1", name="Numeric", seats=1)
await numeric.save()
await Account(id="2", name="Target", seats=1).save()

print("thing:", numeric._get_thing())

await numeric.relate("Membership", "Account:`2`")
related = await numeric.get_related("Membership", model_class=Account)
print("related:", [a.name for a in related])


thing: Account:`1`
related: ['Target']


## 9. Cleanup


In [12]:
for table in ("Account", "Flexible", "Membership"):
    try:
        await client.query(f"REMOVE TABLE {table};")
    except Exception:
        pass

await SurrealDBConnectionManager.close_connection()
print("done")


done
